In [ ]:
import pandas as pd
import requests
import time

# Load your CSV
df = pd.read_csv("coordinates.csv")  # Make sure it has 'decimalLatitude' & 'decimalLongitude'

districts = []

for idx, row in df.iterrows():
    lat = row['decimalLatitude']
    lon = row['decimalLongitude']
    
    # Nominatim reverse geocoding API
    url = f"https://nominatim.openstreetmap.org/reverse?lat={lat}&lon={lon}&format=json&addressdetails=1"
    
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}).json()
        address = response.get('address', {})
        
        # District can be under 'county' or 'district' in OSM
        district = address.get('county') or address.get('district') or ""
        districts.append(district)
    except Exception as e:
        print(f"Error for row {idx}: {e}")
        districts.append("")
    
    time.sleep(1)  # Respect Nominatim rate limit: 1 request per second

# Add the district column to your dataframe
df['district'] = districts

# Save to a new CSV
df.to_csv("coordinates_with_districts.csv", index=False)
print("Done! Saved as coordinates_with_districts.csv")